# BERT Pipeline:
Input text -> Sentence splitting -> BERT embeddings -> Sentence ranking -> Extractive summary

# Explanation:
BERT is an encoder-only transformer model.
Unlike BART, DistilBART and mBART, BERT does not naturally generate summaries.
Therefore, in this notebook BERT is used for extractive summarization:
the most relevant source sentences are selected and combined into a summary.

Work:
Input text  
   ↓  
BERT (Transformers + PyTorch)  
   ↓  
Sentence ranking  
   ↓  
Extractive summary / key content  
   ↓  
spaCy  
   ↓  
Entities: contributors, organizations, dates, etc.

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import json
import time
import re
import spacy

c:\Users\joeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_json("../../Data/silver/doc_01_silver_nlp.json")
print(df.head())

  document_id                                       cleaned_text  \
0      doc_01  Delta Water Innovation Hub\n\nProject ID: DWIH...   

                                              tokens  \
0  [delta, water, innovation, hub, project, id, s...   

                                           sentences  \
0  [Delta Water Innovation Hub\n\nProject ID: DWI...   

                                            entities  
0  [[Delta Water Innovation Hub\n\nProject ID, OR...  


In [3]:
BERT_MODEL = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
model = AutoModel.from_pretrained(BERT_MODEL)
model.eval()

c:\Users\joeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joeri\.cache\huggingface\hub\models--google-bert--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18520.16it/s]
BertModel LOAD

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [4]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def get_bert_embedding(text, max_length=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    return embedding.squeeze(0)

def cosine_similarity(vec1, vec2):
    return torch.nn.functional.cosine_similarity(
        vec1.unsqueeze(0), vec2.unsqueeze(0)
    ).item()

In [5]:
def summarize_bert_extract(text, sentences, top_n=3):
    doc_embedding = get_bert_embedding(text)

    scored_sentences = []
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_embedding = get_bert_embedding(sent)
        score = cosine_similarity(doc_embedding, sent_embedding)
        scored_sentences.append((sent, score))

    scored_sentences = sorted(scored_sentences, key=lambda x: x[1], reverse=True)

    top_sentences = [s for s, _ in scored_sentences[:top_n]]

    # keep original order from source text
    ordered_top_sentences = [s for s in sentences if s in top_sentences]

    summary = " ".join(ordered_top_sentences)
    return summary, scored_sentences

In [6]:
text = df.loc[0, "cleaned_text"]
sentences = df.loc[0, "sentences"]

summary, scored_sentences = summarize_bert_extract(text, sentences, top_n=3)

print(summary)

Delta Water Innovation Hub

Project ID: DWIH-2026-01
Start date: 2026-02-01
End date: 2028-12-31
Contact person: Dr. Lisa Vermeer

The Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland. The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions. The main goal is to reduce pressure on freshwater resources, improve water availability during drought periods and support circular water systems in the southwestern Netherlands.


In [7]:
gold_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "BERT",
    "raw_text": text,
    "summary": summary
}

with open("../Data/gold/BERT/BERT.json", "w") as f:
    json.dump(gold_output, f, indent=4)

print("Saved BERT summary to gold layer.")

Saved BERT summary to gold layer.


In [8]:
def evaluate_summary(original_text, summary_text, source_entities=None):
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)

    compression_ratio = summary_len / original_len if original_len > 0 else 0

    preserved_entities = []
    missed_entities = []

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    for ent in source_entities:
        ent_text = ent[0] if isinstance(ent, (list, tuple)) and len(ent) >= 1 else str(ent)
        ent_text_clean = " ".join(ent_text.split()).strip()

        if ent_text_clean and ent_text_clean.lower() in summary_lower:
            preserved_entities.append(ent_text_clean)
        else:
            missed_entities.append(ent_text_clean)

    total_entities = len(source_entities)
    entity_preservation_score = (
        len(preserved_entities) / total_entities if total_entities > 0 else None
    )

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }

def sentence_coverage_score(source_sentences, summary_text, threshold=0.2):
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue

        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0

In [9]:
start = time.time()
summary, scored_sentences = summarize_bert_extract(
    df.loc[0, "cleaned_text"],
    df.loc[0, "sentences"],
    top_n=3
)
runtime_seconds = time.time() - start

eval_result = evaluate_summary(
    original_text=df.loc[0, "cleaned_text"],
    summary_text=summary,
    source_entities=df.loc[0, "entities"]
)

eval_result["runtime_seconds"] = runtime_seconds
eval_result["sentence_coverage_score"] = sentence_coverage_score(
    df.loc[0, "sentences"], summary
)

print(summary)
print(eval_result)

Delta Water Innovation Hub

Project ID: DWIH-2026-01
Start date: 2026-02-01
End date: 2028-12-31
Contact person: Dr. Lisa Vermeer

The Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland. The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions. The main goal is to reduce pressure on freshwater resources, improve water availability during drought periods and support circular water systems in the southwestern Netherlands.
{'original_token_count': 133, 'summary_token_count': 90, 'compression_ratio': 0.6766917293233082, 'preserved_entities': ['2026-02-01', '2028-12-31', 'Lisa Vermeer', 'Zeeland', 'Lisa Vermeer', 'Netherlands'], 'missed_entities': ['Delta Water Innovation Hub Project ID', 'HZ Kenniscentrum Zeeuwse Samenleving and Water Technology', 'Tom de Bruin'], 'entity_preservation_score': 0.666666666666666

In [10]:
evaluation_row = {
    "document_id": df.loc[0, "document_id"],
    "model": "BERT",
    "runtime_seconds": runtime_seconds,
    "original_token_count": eval_result["original_token_count"],
    "summary_token_count": eval_result["summary_token_count"],
    "compression_ratio": eval_result["compression_ratio"],
    "entity_preservation_score": eval_result["entity_preservation_score"],
    "sentence_coverage_score": eval_result["sentence_coverage_score"]
}

with open("../Data/gold/BERT/BERT_evaluation.json", "w") as f:
    json.dump({
        "document_id": df.loc[0, "document_id"],
        "model": "BERT",
        "evaluation": evaluation_row
    }, f, indent=4)

print("Saved BERT evaluation to gold layer.")

Saved BERT evaluation to gold layer.


In [11]:
nlp = spacy.load("en_core_web_sm")

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]

def extract_people(text):
    doc = nlp(text)
    people = [ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON"]
    return sorted(set(people))

def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text)
    return sorted(set(iso_dates))

def extract_title(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

def extract_description(summary, fallback_text):
    if summary and summary.strip():
        return summary.strip()
    return fallback_text[:500].strip()

def extract_research_groups(text, research_groups):
    text_lower = text.lower()
    matches = []

    for group in research_groups:
        if group.lower() in text_lower:
            matches.append(group)

    return sorted(set(matches))

def extract_project_metadata(text, summary=None, document_id=None):
    people = extract_people(text)
    dates = extract_dates(text)
    research_groups = extract_research_groups(text, RESEARCH_GROUPS)

    metadata = {
        "id": document_id or "",
        "title": extract_title(text),
        "description": extract_description(summary, text),
        "contact_person": people[0] if people else "",
        "start_date": dates[0] if len(dates) > 0 else "",
        "end_date": dates[1] if len(dates) > 1 else "",
        "research_groups": research_groups,
        "researchers": people
    }

    return metadata

In [ ]:
metadata = extract_project_metadata(
    text=df.loc[0, "cleaned_text"],
    summary=summary,
    document_id=df.loc[0, "document_id"]
)

combined_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "BERT",
    "summary": summary,
    "metadata": metadata,
    "evaluation": {
        "runtime_seconds": runtime_seconds,
        "original_token_count": eval_result["original_token_count"],
        "summary_token_count": eval_result["summary_token_count"],
        "compression_ratio": eval_result["compression_ratio"],
        "entity_preservation_score": eval_result["entity_preservation_score"],
        "sentence_coverage_score": eval_result["sentence_coverage_score"]
    }
}

with open("../../Data/gold/BERT/BERT_output.json", "w") as f:
    json.dump(combined_output, f, indent=4)

print("Saved combined BERT output to gold layer.")

Saved combined BERT output to gold layer.
